# Grocery Finder — Data Load (rebuild all Delta tables)

Rebuilds every table the Penny & Parsley app needs from the raw JSONs in
`/Volumes/workspace/grocery_finder/recipe_files/`: `recipes`, `recipe_servings`,
`nofrills_prices`, `superstore_prices`, `safeway_prices`, `walmart_prices`, `iga_prices`.

**Before running:** upload all JSONs (price files + `recipes_export.json` + `servings.json`)
to the volume. If your catalog is not named `workspace`, replace `workspace.` in every cell
(and in `app/grocery_backend.py` CATALOG). Run cells top to bottom on serverless compute.
Early trial-and-error cells from the original notebook were dropped; only working loads +
verification remain.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS grocery_finder;

In [ ]:
%python
# Load the recipe database from recipes_export.json into the recipes Delta table
json_path = "/Volumes/workspace/grocery_finder/recipe_files/recipes_export.json"

# Read JSON into PySpark DataFrame
df_recipes = spark.read.option("multiline", "true").json(json_path)

# Write to Delta Table
df_recipes.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.grocery_finder.recipes")

print("Successfully created Delta table: workspace.grocery_finder.recipes")

In [ ]:
%python
# Build recipe_servings (recipe_id, servings, servings_source) from servings.json.
# Upload data/servings.json into the recipe_files volume first (same folder as the price JSONs).
import json
servings_path = "/Volumes/workspace/grocery_finder/recipe_files/servings.json"
with open(servings_path) as f:
    rows = [(r["recipe_id"], int(r["servings"]), r["servings_source"]) for r in json.load(f)]
servings_df = spark.createDataFrame(rows, "recipe_id string, servings int, servings_source string")
servings_df.write.format("delta").mode("overwrite").saveAsTable("workspace.grocery_finder.recipe_servings")
print(f"recipe_servings created: {servings_df.count()} rows")
display(servings_df.groupBy("servings_source").count())

In [ ]:
%python
# Ingredient price JSON files: (source JSON, target Delta table) — one table per retailer
price_files = [
    ("/Volumes/workspace/grocery_finder/recipe_files/nofrills-366-prices.json", "workspace.grocery_finder.nofrills_prices"),
    ("/Volumes/workspace/grocery_finder/recipe_files/superstore-366-prices (1).json", "workspace.grocery_finder.superstore_prices"),
    ("/Volumes/workspace/grocery_finder/recipe_files/safeway-366-prices (1).json", "workspace.grocery_finder.safeway_prices"),
]

# Read each file, explode its products array into one row per ingredient, and write its own Delta table
for json_path, table_name in price_files:
    df_ingredients = (
        spark.read.option("multiline", "true").json(json_path)
        .selectExpr("retailer", "priceType", "currency", "explode(products) AS product")
        .selectExpr(
            "retailer", "priceType", "currency",
            "product.brand AS brand",
            "product.category AS category",
            "product.ingredient AS ingredient",
            "product.name AS name",
            "CAST(product.packageQuantity AS DOUBLE) AS packageQuantity",  # safeway source is long; cast keeps schemas identical
            "product.packageUnit AS packageUnit",
            "CAST(product.priceCad AS DOUBLE) AS priceCad",
            "product.unitPriceBasis AS unitPriceBasis",
            "CAST(product.unitPriceCad AS DOUBLE) AS unitPriceCad",
        )
    )

    # Write to Delta Table
    df_ingredients.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print(f"Successfully created Delta table: {table_name}")
    display(df_ingredients.limit(5))

In [ ]:
%python
# Retailers whose source JSON has NO brand field (null in every row): Walmart + IGA.
# Files are grouped by target table (all files for one table are combined into that table).
# Brand is derived from the product title so these tables match the nofrills/superstore/safeway tables:
#   1) title starts with a known brand (from the retailers that do have brands + house brands)
#   2) else "Brand - Product" dash pattern (alphabetic first segment, ≤2 words)
#   3) else null (produce/unbranded items — the other tables have nulls there too)
from pyspark.sql import functions as F

retailers_no_brand = [
    ("workspace.grocery_finder.walmart_prices", [
        "/Volumes/workspace/grocery_finder/recipe_files/walmart-364-fresh-prices (1).json",
        "/Volumes/workspace/grocery_finder/recipe_files/walmart-missing-ingredients-prices.json",
    ]),
    ("workspace.grocery_finder.iga_prices", [
        "/Volumes/workspace/grocery_finder/recipe_files/iga-366-prices.json",
    ]),
]

# Brand lexicon: brands from the retailers that have them, plus Walmart/IGA house brands
brands_df = spark.sql("""
    SELECT DISTINCT brand FROM (
        SELECT brand FROM workspace.grocery_finder.nofrills_prices
        UNION ALL SELECT brand FROM workspace.grocery_finder.superstore_prices
        UNION ALL SELECT brand FROM workspace.grocery_finder.safeway_prices
    ) WHERE brand IS NOT NULL
""").unionByName(
    spark.createDataFrame(
        [("Great Value",), ("Your Fresh Market",), ("Kasuku",), ("Applewood Signature",),
         ("Global Quality",), ("Mina Halal",), ("Everland",), ("Kasturi",),
         ("Western Family",)],  # Western Family = IGA house brand
        "brand string",
    )
).distinct().withColumn("brand_lower", F.lower("brand"))

first_segment = F.split(F.col("name"), " - ").getItem(0)

# Guard for the dash rule: in IGA-style titles the first segment is often the product itself
# ("Onions - Yellow - 3Lb"), not a brand. Reject it when it equals or singularizes into the row's ingredient.
seg_lower = F.lower(first_segment)
not_the_ingredient = (
    (F.lower(F.col("ingredient")) != seg_lower)
    & ~F.lower(F.col("ingredient")).contains(F.regexp_replace(seg_lower, "s$", ""))
    & ~F.lower(F.col("ingredient")).contains(F.regexp_replace(seg_lower, "es$", ""))
)

for table_name, source_files in retailers_no_brand:
    # Read each file, explode its products array into one row per ingredient, and combine
    df_walmart = None
    for json_path in source_files:
        df_file = (
            spark.read.option("multiline", "true").json(json_path)
            .selectExpr("retailer", "priceType", "currency", "explode(products) AS product")
            .selectExpr("retailer", "priceType", "currency", "product.*")
        )
        df_walmart = df_file if df_walmart is None else df_walmart.unionByName(df_file)

    # Longest matching brand prefix per title wins (e.g. "Bob's Red Mill" over "Bob's")
    brand_map = (
        df_walmart.select("name")
        .join(F.broadcast(brands_df), F.expr("lower(name) LIKE concat(brand_lower, ' %')"), "left")
        .groupBy("name")
        .agg(F.max_by("brand", F.length("brand")).alias("lexicon_brand"))
    )

    df_walmart = (
        df_walmart
        .join(F.broadcast(brand_map), "name", "left")
        .withColumn("brand", F.coalesce(
            F.col("brand"),                # original source brand if ever present
            F.col("lexicon_brand"),        # rule 1: known-brand prefix
            F.when(
                F.col("name").contains(" - ")
                & first_segment.rlike("^[A-Za-z][A-Za-z ]*$")
                & (F.size(F.split(first_segment, " ")) <= 2)
                & not_the_ingredient,
                first_segment,
            ),  # rule 2: "Brand - Product" dash pattern, but not when the segment is the product itself
        ))
        .drop("lexicon_brand")
        # keep exactly the shared 12-column schema (drops IGA-only fields like estimatedPrice)
        .select("retailer", "priceType", "currency", "brand", "category", "ingredient", "name",
                "packageQuantity", "packageUnit", "priceCad", "unitPriceBasis", "unitPriceCad")
    )

    # Write to Delta Table
    df_walmart.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print(f"Successfully created Delta table: {table_name}")

display(spark.table("workspace.grocery_finder.iga_prices").groupBy("retailer", "brand").count().orderBy(F.desc("count")).limit(15))

In [ ]:
%sql
-- Verification test: row counts, distinct ingredients, and unit-price ranges for all loaded Delta tables
-- Expected: nofrills 514 rows / 278 ingredients, superstore 536 / 285, safeway 435 / 249, iga 481 / 273, walmart 404 / 226, recipes 242 rows
SELECT 'nofrills_prices' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT ingredient) AS distinct_ingredients, ROUND(MIN(unitPriceCad), 6) AS min_unit_price, ROUND(MAX(unitPriceCad), 6) AS max_unit_price FROM workspace.grocery_finder.nofrills_prices
UNION ALL
SELECT 'superstore_prices', COUNT(*), COUNT(DISTINCT ingredient), ROUND(MIN(unitPriceCad), 6), ROUND(MAX(unitPriceCad), 6) FROM workspace.grocery_finder.superstore_prices
UNION ALL
SELECT 'safeway_prices', COUNT(*), COUNT(DISTINCT ingredient), ROUND(MIN(unitPriceCad), 6), ROUND(MAX(unitPriceCad), 6) FROM workspace.grocery_finder.safeway_prices
UNION ALL
SELECT 'iga_prices', COUNT(*), COUNT(DISTINCT ingredient), ROUND(MIN(unitPriceCad), 6), ROUND(MAX(unitPriceCad), 6) FROM workspace.grocery_finder.iga_prices
UNION ALL
SELECT 'walmart_prices', COUNT(*), COUNT(DISTINCT ingredient), ROUND(MIN(unitPriceCad), 6), ROUND(MAX(unitPriceCad), 6) FROM workspace.grocery_finder.walmart_prices
UNION ALL
SELECT 'recipes', COUNT(*), NULL, NULL, NULL FROM workspace.grocery_finder.recipes
ORDER BY table_name;

In [ ]:
%sql
-- Walmart-specific verification: expected 404 rows (365 main run + 39 follow-up batch), 226 distinct ingredients
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT ingredient) AS distinct_ingredients,
       COUNT(DISTINCT category) AS categories,
       COUNT(DISTINCT brand) AS brands,
       ROUND(MIN(unitPriceCad), 6) AS min_unit_price,
       ROUND(MAX(unitPriceCad), 6) AS max_unit_price,
       ROUND(AVG(unitPriceCad), 6) AS avg_unit_price
FROM workspace.grocery_finder.walmart_prices;

In [ ]:
%sql
-- Walmart unit-price basis mix (expected: 310 per g, 78 per mL, 16 per item)
SELECT unitPriceBasis, COUNT(*) AS rows, COUNT(DISTINCT ingredient) AS distinct_ingredients
FROM workspace.grocery_finder.walmart_prices
GROUP BY unitPriceBasis
ORDER BY rows DESC;

In [ ]:
%sql
-- Every distinct ingredient across all 242 recipes, ranked by how many recipes use it
-- Each element of ingredients is a struct {measure, name}; group by the name (case-normalized) so measure variants merge
SELECT lower(trim(ingredient.name)) AS ingredient, COUNT(*) AS recipes_using
FROM (
  SELECT explode(ingredients) AS ingredient
  FROM workspace.grocery_finder.recipes
)
GROUP BY 1
ORDER BY recipes_using DESC, ingredient;

In [ ]:
%sql
-- Sample of the Walmart prices table (same 12-column layout as nofrills/superstore)
SELECT * FROM workspace.grocery_finder.walmart_prices LIMIT 10;

In [ ]:
%sql
-- Sample of the Safeway prices table (same 12-column layout as the other retailers)
SELECT * FROM workspace.grocery_finder.safeway_prices LIMIT 10;

In [ ]:
%python
# Audit: junk brands and coverage in the IGA table, plus schema parity across all price tables
spark.sql("""
    SELECT brand, COUNT(*) AS n FROM workspace.grocery_finder.iga_prices
    WHERE brand IS NOT NULL AND brand RLIKE '[0-9,]'
    GROUP BY brand
""").show()
spark.sql("""
    SELECT COUNT(*) AS rows, SUM(CASE WHEN brand IS NOT NULL THEN 1 ELSE 0 END) AS with_brand, COUNT(DISTINCT brand) AS distinct_brands
    FROM workspace.grocery_finder.iga_prices
""").show()
schemas = {}
for t in ["nofrills_prices", "superstore_prices", "safeway_prices", "iga_prices", "walmart_prices"]:
    rows = spark.sql(f"DESCRIBE workspace.grocery_finder.{t}").collect()
    schemas[t] = [(r[0], r[1]) for r in rows if not r[0].startswith("#")]
    print(t, schemas[t])
print("all five schemas identical:", len(set(map(str, schemas.values()))) == 1)